# PyTorch Refresher


Follows Sebastian Rashcka's excellent [blog post found here](https://sebastianraschka.com/teaching/pytorch-1h).

## Understaning tensors

### Scalars, vectors matrices and tensors

In [1]:
import torch

# 0D tensor (scalar)
tensor0d = torch.tensor(1)

# 1d tensor (vector)
tensor1d = torch.tensor([1, 2, 3])

# 2d tensor (matrix)
tensor2d = torch.tensor([[1, 2], [3, 4]])

# 3d tensor (tensor)
tensor = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]]) 

print(f"Scalar: {tensor0d}")
print(f"Vector: {tensor1d}")
print(f"Matrix: {tensor2d}")
print(f"Tensor: {tensor}")

Scalar: 1
Vector: tensor([1, 2, 3])
Matrix: tensor([[1, 2],
        [3, 4]])
Tensor: tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


### Tensor data types

In [2]:
print(f"Data type: {tensor1d.dtype}")
print(f"Data type: {torch.tensor([1.0, 2.0, 3.0]).dtype}")
print(f"Data type: {tensor1d.to(torch.float32).dtype}")

Data type: torch.int64
Data type: torch.float32
Data type: torch.float32


### Common PyTorch tensor operations

In [3]:
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])

print(f"Tensor shape: {tensor2d.shape}")

Tensor shape: torch.Size([2, 3])


`[2, 3]` means the tensor has 2 rows and 3 columns.

In [4]:
print(f"Tensor reshaped: \n{tensor2d.reshape(3, 2)}")

Tensor reshaped: 
tensor([[1, 2],
        [3, 4],
        [5, 6]])


It is more common to use `.view`. `.reshape` will copy if it cannot use the same memory.

In [5]:
print(f"Tensor reshaped: \n{tensor2d.view(3, 2)}")

Tensor reshaped: 
tensor([[1, 2],
        [3, 4],
        [5, 6]])


The transpose can be taken with `.T`. This is a view.

In [6]:
print(f"Tensor reshaped: \n{tensor2d.T}")

Tensor reshaped: 
tensor([[1, 4],
        [2, 5],
        [3, 6]])


There are two ways of multiplying two matrices.

1. `.matmul`
2. `@` operator

In [7]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [8]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## Seeing models as computation graphs

A computational graph is a direction graph that allows the expression and visualisation of mathematical expressions.
In the context of DL, a computational graph laysout the sequence of calculations neeed to cojmpute the output of a neural network.

![nerual net computational graph](https://sebastianraschka.com/images/teaching/pytorch-1h/figure_07.webp)

In [9]:
import torch.nn.functional as F

y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)

print(f"Input: {x1}")
print(f"Output: {a}")
print(f"Actual: {y}")
print(f"Loss: {loss:.4f}")

Input: tensor([1.1000])
Output: tensor([0.9183])
Actual: tensor([1.])
Loss: 0.0852


## Automatic differentiation made easy

![auto grad on computational graph](https://sebastianraschka.com/images/teaching/pytorch-1h/figure_08.webp)

Gradients are needed when doing backpropagation when training neural networks.

By tracking every operation performed on tensors, PyTorch's autograd engine contructs a computation graph in the background.
When calling the grad function, the gradient of the loss with respect to the model parameter `w1` can be computed.

In [10]:
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
# NOTE: requires_grad has been added
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)


grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(f"Gradient with respect to w1: {grad_L_w1}")
print(f"Gradient with respect to b: {grad_L_b}")

Gradient with respect to w1: (tensor([-0.0898]),)
Gradient with respect to b: (tensor([-0.0817]),)


The above has been done manually, which can be useful for experimentation. In practice, PyTorch provides even more high-level tools to automate this process.
For example, `.backward` can be called on the loss. PyTorch will compute the gradients of all the leaf nodes in the graph, which will be stored via the tensors' `.grad` attributes.

In [11]:
loss.backward()

print(f"Gradient with respect to w1: {w1.grad}")
print(f"Gradient with respect to b: {b.grad}")

Gradient with respect to w1: tensor([-0.0898])
Gradient with respect to b: tensor([-0.0817])


## Implementing mulilayer neural networks

When implementing neural networks in PyTorch, typically `torch.nn.Module` is subclassed.
This `Module` base class provides a lot of functionality. For example, it allows the encapsulation of layers and operations and keep track of the model's performance.

Within the subclass, the network layers are defined in the `__init__` constructor and specify how they interact in the `forward` method.
The `forward` method describes how the input data passes through the network and comes together as a computational graph.

The backwards mehtod is not typically needed to be implemented.

In [12]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_ouputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_ouputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [13]:
model = NeuralNetwork(num_inputs=50, num_ouputs=3)
print(f"Model: {model}")

Model: NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Note that using `Sequential` means that only `self.layers` needs to be called instead of calling each layer individualy.

In [14]:
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Total num of trainable model parameters: {num_params}")

Total num of trainable model parameters: 2213


Note that each param for which `requires_grad=True` counts as a trainable parameter and will be updated during training.

A *linear layer* multiplies the inputs with a weight matrix and adds a bias vector. This is sometimes referred to as a *feedforward* or *full connected layer*.

The corresponding weight parmater matrix can be accessed as follows:

In [15]:
print(f"First layer weights: {model.layers[0].weight}")
print(f"First layer weights shape: {model.layers[0].weight.shape}")

First layer weights: Parameter containing:
tensor([[ 0.1252,  0.0133,  0.0487,  ...,  0.0406, -0.0348, -0.0806],
        [-0.0632, -0.1159, -0.0201,  ..., -0.1034,  0.1407, -0.0069],
        [ 0.0399, -0.0447, -0.0891,  ...,  0.0852, -0.0463,  0.0996],
        ...,
        [ 0.0179,  0.0213, -0.1028,  ..., -0.0790,  0.0328,  0.0174],
        [-0.1280,  0.0686,  0.0063,  ...,  0.1368, -0.0296,  0.1255],
        [-0.0177, -0.0375,  0.0927,  ..., -0.0700, -0.1040, -0.0651]],
       requires_grad=True)
First layer weights shape: torch.Size([30, 50])


The bias can be accessed like this:

In [16]:
print(f"First layer biases: \n{model.layers[0].bias}")
print(f"First layer biases shape: {model.layers[0].bias.shape}")

First layer biases: 
Parameter containing:
tensor([-0.0417, -0.0035, -0.1299,  0.1058,  0.1338, -0.0419,  0.1178, -0.0868,
         0.1233, -0.0613, -0.0127,  0.0797, -0.1303, -0.0230,  0.0196,  0.1223,
         0.0098, -0.0736, -0.1167,  0.1124, -0.0859, -0.1299,  0.0028,  0.1275,
        -0.0322, -0.1313,  0.1050,  0.1395, -0.0301, -0.0130],
       requires_grad=True)
First layer biases shape: torch.Size([30])


These numbers are different each time, as they are randomised. To control this, use `torch.manual_seed`.

In [17]:
torch.manual_seed(123)

model = NeuralNetwork(num_inputs=50, num_ouputs=3)

print(f"First layer weights: {model.layers[0].weight}")
print(f"First layer weights shape: {model.layers[0].weight.shape}")

First layer weights: Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)
First layer weights shape: torch.Size([30, 50])


To use the nerual network via the foward pass:

In [18]:
torch.manual_seed(123)

X = torch.rand((1, 50))
out = model(X)

print(f"Input: {X}")
print(f"Output logits: {out}")

Input: tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])
Output logits: tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


When calling `model(X)`, the forward pass of the model is automatically called.

`grad_fn=<AddmmBackward0>` represents the last-used function to compute a variable in the computational graph. PyTorch will use this informaiton when it computes gradients during backpropagation.
`Addmm` stands for matrix multiplication, followed by an additon (`Add`)

If using the network without training or backpropagation, for example, just for prediction after training, contricuting this computational graph for backpropagation can be wasteful. This is because it performs unnecessary computations and consumes additional memory. So it's best practice to use `torch.no_grad()`. This tells PyTorch taht it doesn't need to keep track of the gradients. This can be a significant saving in memory and computation.

In [19]:
with torch.no_grad():
    out = model(X)

print(f"Output logits: {out}")

Output logits: tensor([[-0.1262,  0.1080, -0.1792]])


In PyTorch, it's common to code models so that they return the outputs of the last layer (`logits`) without passing them to a nonlinear activation function.
That's because PyTorch's commonly used loss functions combine the softmax (or sigmoid for binary classification) operation with the negative log-likelihood loss in a single class.

The reason for this is numerical efficiency and stability. Because of this, the softmax function has to be called explicitly:

In [20]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)

print(f"Output: {out}")

Output: tensor([[0.3113, 0.3934, 0.2952]])


The values can now be interpreted as class-membership probs that sum up to 1.

## Setting up efficient data loaders

![data loading](https://sebastianraschka.com/images/teaching/pytorch-1h/figure_10.webp)

In [21]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [23]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

### Class label numbering

PyTorch requires that class labels start with label 0 and the largest class label should not exceed the number of output nodes minus 1.

In [26]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
tet_ds = ToyDataset(X_test, y_test)

In PyTorch, the three main components of a custom `Dataset` class are the `__init__` constructor and `__getitem__`/`__len__` methods.

In the `__getitem__` method, the instructions for returning exactly one item from the dataset via an index is given. This means the features for the class label corresponding to a signle training example or test instance.

In [27]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
)

test_ds = ToyDataset(X_test, y_test)
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

In [29]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx + 1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


Note that with a batch size of 2, the 3rd batch only contains a single example. In practice, hnaving a substaially smaller batch as the last batch in a training epoch can disturb the convergence during training. To prevent this, it's recommended to set `drop_last=True`.

## A typical training loop

In [35]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_ouputs=2)
optimiser = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    print()

Epoch: 001/003 | Batch 000/003 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/003 | Train/Val Loss: 0.65
Epoch: 001/003 | Batch 002/003 | Train/Val Loss: 0.42

Epoch: 002/003 | Batch 000/003 | Train/Val Loss: 0.05
Epoch: 002/003 | Batch 001/003 | Train/Val Loss: 0.13
Epoch: 002/003 | Batch 002/003 | Train/Val Loss: 0.00

Epoch: 003/003 | Batch 000/003 | Train/Val Loss: 0.01
Epoch: 003/003 | Batch 001/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 002/003 | Train/Val Loss: 0.02



`model.train` and `model.eval` are new settings. They put the model into training and eval mode. This is necessary for components that behave differently during training and inference, such as dropout or batch normalization layers.

`loss.backward` calculated the gradients in the computational graph and `optimiser.step` used the gradients to update themodel parameters to minimise the loss.
In the case of SGD, the means multiplying the gradients with the learning rate and adding the scaled negative gradient to the parameters.

It is also important to include `optimiser.zero_grad()` in each round to reset the gradients to zero. Without this, the gradients would accumlate, which is undesired.

In [41]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)

torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
preds = torch.argmax(probas, dim=1)
results = preds == y_test

print(f"Test logits: {outputs}")
print(f"Test outputs: {probas}")
print(f"Test preds: {preds}")
print(f"Test results: {results}")

Test logits: tensor([[ 2.4715, -3.6645],
        [-2.4984,  2.5158]])
Test outputs: tensor([[0.9978, 0.0022],
        [0.0066, 0.9934]])
Test preds: tensor([0, 1])
Test results: tensor([True, True])


## Saving and loading models

In [42]:
torch.save(model.state_dict(), "model.pth")

`.pth` and `.pt` are the most common file name conventions.

Once the model is saved, it can be restored from disk as follows:

In [43]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>